# 23b · Graph-assisted retrieval with explicit provenance

## The engineering question
When a question follows relationships, can a small explicit graph retrieve evidence that a similarity budget misses?

**Learning outcomes:** build source-backed edges; seed traversal with vector search; enforce access before expanding edges; compare dense retrieval, graph expansion, and exact structured lookup; measure evidence coverage and real generation cost.

This is a fictional supply-chain fixture with hand-authored edges. Graph construction is deterministic author work, not LLM extraction. Qdrant finds seed documents; NetworkX traverses a separate graph. Three source passages support one multi-hop answer. We measure a top-two dense baseline against a graph with at most six edges and depth three, exposing the larger evidence budget. A known structured identifier also supports a relational lookup without embeddings.

Microsoft GraphRAG additionally describes LLM extraction and community summaries for corpus-wide questions. That preprocessing has extraction, entity-resolution, update, provenance, and summary-generation costs. This local traversal example does not reproduce those summaries or the paper's large-corpus results. [GraphRAG paper](https://arxiv.org/abs/2404.16130v2).

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("23b", "O", docker=True)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
import time
import sqlite3
import networkx as nx
from uuid import uuid4
from transformers import AutoTokenizer
from vector_masterclass.config import model_path
from vector_masterclass.schema import Document
from vector_masterclass.chunking import chunk_document
from vector_masterclass.store import VectorIndex
from vector_masterclass.context import assemble
from vector_masterclass.models import release_models
from vector_masterclass.rag import answer
from vector_masterclass.evidence import json_write

records = [
    ("project", "Orion project uses component Kestrel."),
    ("component", "Kestrel component is supplied by Northstar."),
    ("supplier", "Northstar supplier operates in Harbor City."),
    ("distractor", "Orion project launch notes discuss software testing and schedules."),
    ("other", "Lumen project uses component Robin."),
    ("private", "Northstar has confidential internal account 8821."),
]
docs = [
    Document(
        id=id_, revision="1", title=id_, text=text, tenant="red" if id_ == "private" else "public"
    )
    for id_, text in records
]
edges = [
    ("Orion", "Kestrel", "project"),
    ("Kestrel", "Northstar", "component"),
    ("Northstar", "Harbor City", "supplier"),
    ("Lumen", "Robin", "other"),
    ("Northstar", "8821", "private"),
]
by_id = {d.id: d for d in docs}
graph = nx.DiGraph()
for source, target, doc_id in edges:
    graph.add_edge(source, target, source_id=doc_id, tenant=by_id[doc_id].tenant)
sql = sqlite3.connect(":memory:")
sql.execute("create table edge(source text,target text,source_id text,tenant text)")
sql.executemany(
    "insert into edge values(?,?,?,?)", [(a, b, c, by_id[c].tenant) for a, b, c in edges]
)
question = "Which city does the supplier of the component used by Orion operate in?"
gold = {"project", "component", "supplier"}
index = VectorIndex("vmc_23b_" + uuid4().hex[:8])
index.create()
started = time.perf_counter()
try:
    for d in docs:
        index.upsert(chunk_document(d, "document"), document_id=d.id, include_sparse=False)
    hits = index.search(question, tenant="public", mode="dense", k=2, exact=True)
    dense_ids = [h.chunk.document_key.split("@")[0] for h in hits]
finally:
    index.drop()
    index.close()
retrieval_seconds = time.perf_counter() - started
# Authorized relation sources only. Never traverse a private edge then redact at the end.
eligible = nx.DiGraph(
    [(a, b, data) for a, b, data in graph.edges(data=True) if data["tenant"] == "public"]
)
seeds = sorted({a for a, b, data in eligible.edges(data=True) if data["source_id"] in dense_ids})
started = time.perf_counter()
visited_edges = []
frontier = seeds
for depth in range(3):
    next_frontier = []
    for source in frontier:
        for target in sorted(eligible.successors(source)):
            edge = (source, target)
            if edge not in visited_edges and len(visited_edges) < 6:
                visited_edges.append(edge)
                next_frontier.append(target)
    frontier = next_frontier
graph_ids = list(dict.fromkeys(dense_ids + [eligible.edges[e]["source_id"] for e in visited_edges]))
traversal_seconds = time.perf_counter() - started
structured = sql.execute(
    "select c.target from edge a join edge b on a.target=b.source join edge c on b.target=c.source where a.source=? and a.tenant='public' and b.tenant='public' and c.tenant='public'",
    ("Orion",),
).fetchall()
sql.close()
tokenizer = AutoTokenizer.from_pretrained(str(model_path("generator")), local_files_only=True)
prepared = []
for method, selected in [("dense top 2", dense_ids), ("dense + bounded graph", graph_ids)]:
    context = assemble(
        [chunk_document(by_id[id_], "document")[0] for id_ in selected],
        docs,
        tokenizer,
        384,
        tenant="public",
    )
    prepared.append((method, selected, context))
release_models()
rows = []
traces = []
for method, selected, context in prepared:
    generated = answer(question, context)
    traces.append(generated | {"method": method})
    rows.append(
        {
            "method": method,
            "documents": len(selected),
            "gold_source_coverage": len(gold & set(selected)) / len(gold),
            "evidence_tokens": context["tokens"],
            "generation_seconds": generated["seconds"],
            "answer_mentions_city": "harbor city" in generated["text"].lower(),
            "citations_resolve": generated["citation_audit"]["all_resolve"],
            "answer": generated["text"],
        }
    )
results = lesson.table(pd.DataFrame(rows))
display(results)
print("Exact structured lookup from known Orion ID:", structured)
print("Vector seeds:", seeds, "traversed edges:", visited_edges)
json_write(
    lesson.path / "graph-and-traces.json",
    {
        "edges": edges,
        "construction": "author-curated fictional facts",
        "traces": traces,
        "selected_sources": graph_ids,
    },
)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
nx.draw_networkx(
    eligible,
    pos=nx.spring_layout(eligible, seed=17),
    ax=axes[0],
    node_size=1200,
    font_size=8,
    node_color="#c9def0",
    arrows=True,
)
axes[0].set_title("Authorized source-backed relation graph")
axes[0].axis("off")
results.set_index("method")[
    ["gold_source_coverage", "answer_mentions_city", "citations_resolve"]
].astype(float).plot.bar(ax=axes[1], ylim=(0, 1.05))
axes[1].tick_params(axis="x", rotation=12)
axes[1].legend(fontsize=7)
fig.tight_layout()
lesson.figure(fig)
plt.show()

## Interpretation and exercise
More retrieved evidence costs tokens and can introduce errors. A city keyword and valid citation IDs do not prove that the generated relationship chain is correct; inspect each saved answer. Missing seed entities, false merged entities, reversed edges, high-degree hubs, and stale or unauthorized edges can make graph expansion worse. Community summaries answer broader questions but must retain traceable supporting sources and be regenerated or invalidated when those sources change.

**Exercise:** remove the Kestrel-to-Northstar edge, rerun, and explain why extra hops cannot invent the missing relation. Add a competing supplier with a dated contract and define temporal validity before traversal. Compare SQL for the known identifier with graph-assisted natural-language retrieval; explain when neither needs a vector database.

In [ ]:
manifest = lesson.finish(
    {
        "real_vector_seed": len(dense_ids) == 2,
        "bounded_traversal": len(visited_edges) <= 6,
        "private_edge_excluded": "private" not in graph_ids and "8821" not in eligible,
        "structured_lookup": structured == [("Harbor City",)],
        "two_live_generations": all(t["live"] for t in traces),
    },
    observations={
        "retrieval_including_ingestion_seconds": retrieval_seconds,
        "traversal_seconds": traversal_seconds,
        "graph_construction": "hand-authored",
        "LLM_extraction_executed": False,
        "community_summaries_executed": False,
        "dense_top_k": 2,
        "graph_max_edges": 6,
        "graph_max_depth": 3,
    },
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.